In [16]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np
import pickle

# Set seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)

# ==========================================
# 1. LOAD ACTUAL DATA
# ==========================================
print("Loading actual data from pickles...")

# Load binary success rate (y_data)
with open('data/resmat_binary_success_rate.pkl', 'rb') as f:
    y_data = pickle.load(f)
    # Convert DataFrame to numpy if needed
    if hasattr(y_data, 'values'):
        y_data = y_data.values
    # Ensure numeric type and convert to float32
    y_data = np.array(y_data, dtype=np.float32)
    # Replace NaN with 0.5 (neutral value for binary data)
    y_data = np.nan_to_num(y_data, nan=0.5)
    y_data = torch.from_numpy(y_data)

# Load z_data (behavioral attributes)
z_data_list = []
z_names = ['environmentalbarrier', 'instructionfollowing', 'selfcorrection', 'tooluse', 'verification']

for z_name in z_names:
    with open(f'data/resmat_{z_name}.label.pkl', 'rb') as f:
        z_matrix = pickle.load(f)
        # Convert DataFrame to numpy if needed
        if hasattr(z_matrix, 'values'):
            z_matrix = z_matrix.values
        # Ensure numeric type and convert to float32
        z_matrix = np.array(z_matrix, dtype=np.float32)
        # Replace NaN with 0.5 (neutral value for binary data)
        z_matrix = np.nan_to_num(z_matrix, nan=0.5)
        z_data_list.append(torch.from_numpy(z_matrix))

# Stack z_data: (N, J, M)
z_data = torch.stack(z_data_list, dim=2)

# Get dimensions from data
N, J = y_data.shape
M = z_data.shape[2]

# Generate random item features (since we only have resmat_* files)
# Using J items and a reasonable feature dimension
d_features = 10
x_j_raw = torch.randn(J, d_features)
x_j_input = F.normalize(x_j_raw, p=2, dim=1)

print(f"Data loaded: N={N} (models), J={J} (items), M={M} (behaviors), d_features={d_features}")
print(f"y_data shape: {y_data.shape}")
print(f"z_data shape: {z_data.shape}")
print(f"x_j shape: {x_j_input.shape}")

# ==========================================
# 2. ROBUST MODEL (ReLU + Normalized W)
# ==========================================
class RobustARDModel(nn.Module):
    def __init__(self, N, J, M, K_model, d_features, x_j_input):
        super().__init__()
        self.N, self.J, self.M, self.K = N, J, M, K_model

        # Register fixed item features
        self.register_buffer('x_j', x_j_input)

        # Parameters
        self.theta = nn.Parameter(torch.randn(N, K_model) * 0.1)
        self.W = nn.Parameter(torch.randn(K_model, d_features) * 0.1)

        # Tau: Initialize to 0.5 so they start alive
        self.tau_raw = nn.Parameter(torch.ones(K_model) * 0.5)

        self.u_logits = nn.Parameter(torch.ones(M, K_model) * 2.0)
        self.delta_j = nn.Parameter(torch.zeros(J))
        self.delta_zm = nn.Parameter(torch.zeros(J, M))

    @property
    def tau(self):
        # ReLU ensures exact zeros (sparsity)
        return F.relu(self.tau_raw)

    def get_gates(self, temp):
        return torch.sigmoid(self.u_logits / temp)

    def forward(self, temp=1.0):
        # Normalize W so scale is handled purely by tau
        W_norm = F.normalize(self.W, dim=1)

        # Amortized loadings
        base_loadings = self.x_j @ W_norm.T
        a_j = base_loadings * self.tau.unsqueeze(0)

        g_m = self.get_gates(temp)

        # Overall prediction
        logits_y = self.theta @ a_j.T + self.delta_j.unsqueeze(0)

        # Subskill prediction
        logits_z_list = []
        for m in range(self.M):
            a_masked = a_j * g_m[m].unsqueeze(0)
            l_z = self.theta @ a_masked.T + self.delta_zm[:, m].unsqueeze(0)
            logits_z_list.append(l_z.unsqueeze(2))

        return logits_y, torch.cat(logits_z_list, dim=2)

# ==========================================
# 3. OPTIMIZATION LOOP
# ==========================================

K_MODEL = 25
model = RobustARDModel(N, J, M, K_MODEL, d_features, x_j_input)

# Separate parameter groups:
# We generally want a smaller LR for the structure (tau) to prevent oscillation
optimizer = optim.Adam([
    {'params': model.tau_raw, 'lr': 0.005},  # Slower learning for ARD
    {'params': [p for n, p in model.named_parameters() if 'tau' not in n], 'lr': 0.01}
])

# CRITICAL: Higher Lambda to overcome N*J likelihood sum
# Rule of thumb: Lambda ~= 1.5 * N often works for factor models
# Adjusted for smaller dataset size (N=46)
hyperparams = {'lambda_tau': 5.0}

print(f"\nStarting Robust ARD with K_model={K_MODEL}...")
print(f"Using Lambda Tau: {hyperparams['lambda_tau']}")

for e in range(1001):
    optimizer.zero_grad()

    # Forward
    logits_y, logits_z = model(temp=1.0)

    # Loss: Likelihood (Sum reduction scales with N*J)
    loss_lik = F.binary_cross_entropy_with_logits(logits_y, y_data, reduction='sum') + \
               F.binary_cross_entropy_with_logits(logits_z, z_data, reduction='sum')

    # Loss: L1 Penalty on Tau
    reg_tau = hyperparams['lambda_tau'] * torch.norm(model.tau, 1)

    # Loss: Regularization on other params
    reg_theta = 0.5 * torch.sum(model.theta ** 2)
    reg_W = 0.5 * torch.sum(model.W ** 2)

    loss = loss_lik + reg_tau + reg_theta + reg_W
    loss.backward()
    optimizer.step()

    # --- SNAPPING TRICK ---
    # If a dimension is very small, kill it to allow ReLU to keep it dead.
    with torch.no_grad():
        # If tau < 0.01, set the raw parameter to -0.1 (dead ReLU region)
        small_indices = model.tau < 0.01
        model.tau_raw[small_indices] = -0.1

    if e % 100 == 0:
        current_tau = model.tau.detach().numpy()
        print(f"Ep {e} | Loss {loss.item():.2e} | Tau: {np.round(current_tau, 3)}")

# ==========================================
# 4. FINAL RESULTS
# ==========================================
print("\n--- FINAL SCALES ---")
final_tau = model.tau.detach().numpy()
sorted_tau = np.sort(final_tau)[::-1]

print(f"Sorted Tau: {np.round(sorted_tau, 3)}")

# Count effective dimensions (strictly positive)
effective_k = np.sum(final_tau > 0.0)
print(f"Effective Dimension: {effective_k}")
print(f"\nBehavior names: {z_names}")
print(f"Gates (u_logits) shape: {model.u_logits.shape}")

Loading actual data from pickles...
Data loaded: N=46 (models), J=191 (items), M=5 (behaviors), d_features=10
y_data shape: torch.Size([46, 191])
z_data shape: torch.Size([46, 191, 5])
x_j shape: torch.Size([191, 10])

Starting Robust ARD with K_model=25...
Using Lambda Tau: 5.0
Ep 0 | Loss 3.66e+04 | Tau: [0.495 0.495 0.495 0.495 0.495 0.495 0.495 0.495 0.495 0.495 0.495 0.495
 0.495 0.495 0.495 0.495 0.495 0.495 0.495 0.495 0.495 0.495 0.495 0.495
 0.495]


/tmp/ipykernel_6809/3174479986.py:19: DeprecationWarning: numpy.core.numeric is deprecated and has been renamed to numpy._core.numeric. The numpy._core namespace contains private NumPy internals and its use is discouraged, as NumPy internals can change without warning in any release. In practice, most real-world usage of numpy.core is to access functionality in the public NumPy API. If that is the case, use the public NumPy API. If not, you are using NumPy internals. If you would still like to access an internal attribute, use numpy._core.numeric._frombuffer.
  y_data = pickle.load(f)
/tmp/ipykernel_6809/3174479986.py:35: DeprecationWarning: numpy.core.numeric is deprecated and has been renamed to numpy._core.numeric. The numpy._core namespace contains private NumPy internals and its use is discouraged, as NumPy internals can change without warning in any release. In practice, most real-world usage of numpy.core is to access functionality in the public NumPy API. If that is the case, u

Ep 100 | Loss 3.57e+04 | Tau: [0.154 0.139 0.189 0.198 0.183 0.146 0.176 0.193 0.142 0.185 0.135 0.192
 0.18  0.204 0.162 0.158 0.187 0.176 0.207 0.189 0.165 0.129 0.205 0.142
 0.171]
Ep 200 | Loss 3.57e+04 | Tau: [0.196 0.    0.359 0.333 0.023 0.    0.225 0.28  0.    0.443 0.    0.49
 0.369 0.589 0.    0.396 0.336 0.036 0.245 0.532 0.08  0.    0.822 0.
 0.01 ]
Ep 300 | Loss 3.57e+04 | Tau: [0.379 0.    0.546 0.489 0.    0.    0.353 0.2   0.    0.924 0.    0.814
 0.423 0.849 0.    0.786 0.464 0.    0.195 0.819 0.    0.    1.115 0.
 0.   ]
Ep 400 | Loss 3.56e+04 | Tau: [0.528 0.    0.772 0.254 0.    0.    0.478 0.117 0.    1.181 0.    0.82
 0.339 0.875 0.    0.9   0.599 0.    0.23  0.765 0.    0.    1.316 0.
 0.   ]
Ep 500 | Loss 3.56e+04 | Tau: [0.537 0.    0.916 0.336 0.    0.    0.511 0.    0.    1.307 0.    0.934
 0.338 0.926 0.    0.856 0.627 0.    0.346 0.577 0.    0.    1.509 0.
 0.   ]
Ep 600 | Loss 3.56e+04 | Tau: [0.53  0.    0.944 0.412 0.    0.    0.522 0.    0.    1.343 0. 

In [17]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np
import pickle
import pandas as pd

# Set seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)

# ==========================================
# 1. LOAD ACTUAL DATA
# ==========================================
print("Loading actual data from pickles...")

# Load binary success rate (y_data)
with open('data/resmat_binary_success_rate.pkl', 'rb') as f:
    y_data = pickle.load(f)
    # Convert DataFrame to numpy if needed
    if hasattr(y_data, 'values'):
        y_data = y_data.values
    # Ensure numeric type and convert to float32
    y_data = np.array(y_data, dtype=np.float32)
    # Replace NaN with 0.5 (neutral value for binary data)
    y_data = np.nan_to_num(y_data, nan=0.5)
    y_data = torch.from_numpy(y_data)

# Load z_data (behavioral attributes)
z_data_list = []
z_names = ['environmentalbarrier', 'instructionfollowing', 'selfcorrection', 'tooluse', 'verification']

for z_name in z_names:
    with open(f'data/resmat_{z_name}.label.pkl', 'rb') as f:
        z_matrix = pickle.load(f)
        # Convert DataFrame to numpy if needed
        if hasattr(z_matrix, 'values'):
            z_matrix = z_matrix.values
        # Ensure numeric type and convert to float32
        z_matrix = np.array(z_matrix, dtype=np.float32)
        # Replace NaN with 0.5 (neutral value for binary data)
        z_matrix = np.nan_to_num(z_matrix, nan=0.5)
        z_data_list.append(torch.from_numpy(z_matrix))

# Stack z_data: (N, J, M)
z_data = torch.stack(z_data_list, dim=2)

# Load item features from embeddings mapping
with open('data/task_id_to_embedding.pkl', 'rb') as f:
    task_to_emb = pickle.load(f)
    
print(f"Loaded embeddings for {len(task_to_emb)} task_ids")

# Get column MultiIndex from y_data to match order
with open('data/resmat_binary_success_rate.pkl', 'rb') as f:
    y_df_original = pickle.load(f)
    if hasattr(y_df_original, 'columns'):
        # Extract task_ids from MultiIndex level 0
        task_ids_ordered = y_df_original.columns.get_level_values(0).tolist()
    else:
        raise ValueError("Cannot extract task IDs from resmat")

# Match embeddings to items in order
embeddings_list = []
matched_count = 0
for task_id in task_ids_ordered:
    if task_id in task_to_emb:
        embeddings_list.append(task_to_emb[task_id])
        matched_count += 1
    else:
        # If no embedding found, use zero vector
        print(f"Warning: No embedding for task_id {task_id}, using zero vector")
        embeddings_list.append(np.zeros(2560, dtype=np.float32))

x_j_input = np.array(embeddings_list, dtype=np.float32)
x_j_input = torch.from_numpy(x_j_input)
# Normalize features
x_j_input = F.normalize(x_j_input, p=2, dim=1)

# Get dimensions from data
N, J = y_data.shape
M = z_data.shape[2]
d_features = x_j_input.shape[1]

print(f"Data loaded: N={N} (models), J={J} (items), M={M} (behaviors), d_features={d_features}")
print(f"Matched {matched_count}/{J} embeddings")
print(f"y_data shape: {y_data.shape}")
print(f"z_data shape: {z_data.shape}")
print(f"x_j shape: {x_j_input.shape}")

# ==========================================
# 2. ROBUST MODEL (ReLU + Normalized W)
# ==========================================
class RobustARDModel(nn.Module):
    def __init__(self, N, J, M, K_model, d_features, x_j_input):
        super().__init__()
        self.N, self.J, self.M, self.K = N, J, M, K_model

        # Register fixed item features
        self.register_buffer('x_j', x_j_input)

        # Parameters
        self.theta = nn.Parameter(torch.randn(N, K_model) * 0.1)
        self.W = nn.Parameter(torch.randn(K_model, d_features) * 0.1)

        # Tau: Initialize to 0.5 so they start alive
        self.tau_raw = nn.Parameter(torch.ones(K_model) * 0.5)

        self.u_logits = nn.Parameter(torch.ones(M, K_model) * 2.0)
        self.delta_j = nn.Parameter(torch.zeros(J))
        self.delta_zm = nn.Parameter(torch.zeros(J, M))

    @property
    def tau(self):
        # ReLU ensures exact zeros (sparsity)
        return F.relu(self.tau_raw)

    def get_gates(self, temp):
        return torch.sigmoid(self.u_logits / temp)

    def forward(self, temp=1.0):
        # Normalize W so scale is handled purely by tau
        W_norm = F.normalize(self.W, dim=1)

        # Amortized loadings
        base_loadings = self.x_j @ W_norm.T
        a_j = base_loadings * self.tau.unsqueeze(0)

        g_m = self.get_gates(temp)

        # Overall prediction
        logits_y = self.theta @ a_j.T + self.delta_j.unsqueeze(0)

        # Subskill prediction
        logits_z_list = []
        for m in range(self.M):
            a_masked = a_j * g_m[m].unsqueeze(0)
            l_z = self.theta @ a_masked.T + self.delta_zm[:, m].unsqueeze(0)
            logits_z_list.append(l_z.unsqueeze(2))

        return logits_y, torch.cat(logits_z_list, dim=2)

# ==========================================
# 3. OPTIMIZATION LOOP
# ==========================================

K_MODEL = 25
model = RobustARDModel(N, J, M, K_MODEL, d_features, x_j_input)

# Separate parameter groups:
# We generally want a smaller LR for the structure (tau) to prevent oscillation
optimizer = optim.Adam([
    {'params': model.tau_raw, 'lr': 0.005},  # Slower learning for ARD
    {'params': [p for n, p in model.named_parameters() if 'tau' not in n], 'lr': 0.01}
])

# CRITICAL: Higher Lambda to overcome N*J likelihood sum
# Rule of thumb: Lambda ~= 1.5 * N often works for factor models
# Adjusted for smaller dataset size (N=46)
hyperparams = {'lambda_tau': 5.0}

print(f"\nStarting Robust ARD with K_model={K_MODEL}...")
print(f"Using Lambda Tau: {hyperparams['lambda_tau']}")

for e in range(1001):
    optimizer.zero_grad()

    # Forward
    logits_y, logits_z = model(temp=1.0)

    # Loss: Likelihood (Sum reduction scales with N*J)
    loss_lik = F.binary_cross_entropy_with_logits(logits_y, y_data, reduction='sum') + \
               F.binary_cross_entropy_with_logits(logits_z, z_data, reduction='sum')

    # Loss: L1 Penalty on Tau
    reg_tau = hyperparams['lambda_tau'] * torch.norm(model.tau, 1)

    # Loss: Regularization on other params
    reg_theta = 0.5 * torch.sum(model.theta ** 2)
    reg_W = 0.5 * torch.sum(model.W ** 2)

    loss = loss_lik + reg_tau + reg_theta + reg_W
    loss.backward()
    optimizer.step()

    # --- SNAPPING TRICK ---
    # If a dimension is very small, kill it to allow ReLU to keep it dead.
    with torch.no_grad():
        # If tau < 0.01, set the raw parameter to -0.1 (dead ReLU region)
        small_indices = model.tau < 0.01
        model.tau_raw[small_indices] = -0.1

    if e % 100 == 0:
        current_tau = model.tau.detach().numpy()
        print(f"Ep {e} | Loss {loss.item():.2e} | Tau: {np.round(current_tau, 3)}")

# ==========================================
# 4. FINAL RESULTS
# ==========================================
print("\n--- FINAL SCALES ---")
final_tau = model.tau.detach().numpy()
sorted_tau = np.sort(final_tau)[::-1]

print(f"Sorted Tau: {np.round(sorted_tau, 3)}")

# Count effective dimensions (strictly positive)
effective_k = np.sum(final_tau > 0.0)
print(f"Effective Dimension: {effective_k}")
print(f"\nBehavior names: {z_names}")
print(f"Gates (u_logits) shape: {model.u_logits.shape}")


Loading actual data from pickles...
Loaded embeddings for 156 task_ids
Data loaded: N=46 (models), J=191 (items), M=5 (behaviors), d_features=2560
Matched 191/191 embeddings
y_data shape: torch.Size([46, 191])
z_data shape: torch.Size([46, 191, 5])
x_j shape: torch.Size([191, 2560])

Starting Robust ARD with K_model=25...
Using Lambda Tau: 5.0
Ep 0 | Loss 3.69e+04 | Tau: [0.495 0.495 0.495 0.495 0.495 0.495 0.495 0.495 0.495 0.495 0.495 0.495
 0.495 0.495 0.495 0.495 0.495 0.495 0.495 0.495 0.495 0.495 0.495 0.495
 0.495]


/tmp/ipykernel_6809/4272501589.py:20: DeprecationWarning: numpy.core.numeric is deprecated and has been renamed to numpy._core.numeric. The numpy._core namespace contains private NumPy internals and its use is discouraged, as NumPy internals can change without warning in any release. In practice, most real-world usage of numpy.core is to access functionality in the public NumPy API. If that is the case, use the public NumPy API. If not, you are using NumPy internals. If you would still like to access an internal attribute, use numpy._core.numeric._frombuffer.
  y_data = pickle.load(f)
/tmp/ipykernel_6809/4272501589.py:36: DeprecationWarning: numpy.core.numeric is deprecated and has been renamed to numpy._core.numeric. The numpy._core namespace contains private NumPy internals and its use is discouraged, as NumPy internals can change without warning in any release. In practice, most real-world usage of numpy.core is to access functionality in the public NumPy API. If that is the case, u

Ep 100 | Loss 3.52e+04 | Tau: [0.35  0.457 0.313 0.765 0.519 0.563 0.824 0.499 0.587 0.651 0.607 0.519
 0.252 0.528 0.793 0.54  0.613 0.554 0.691 0.546 0.509 0.828 0.544 0.562
 0.752]
Ep 200 | Loss 3.46e+04 | Tau: [0.376 0.637 0.289 1.287 0.653 0.681 1.606 0.643 0.936 0.768 0.707 0.789
 0.245 0.638 1.106 0.955 0.712 0.734 0.976 0.842 0.654 1.182 0.67  0.672
 0.92 ]
Ep 300 | Loss 3.44e+04 | Tau: [0.362 0.789 0.324 1.549 0.785 0.73  2.004 0.711 1.101 0.82  0.791 1.128
 0.302 0.783 1.281 1.28  0.751 0.89  1.108 1.087 0.722 1.251 0.788 0.714
 1.077]
Ep 400 | Loss 3.42e+04 | Tau: [0.329 0.956 0.47  1.725 0.868 0.764 2.234 0.813 1.221 0.849 0.91  1.612
 0.322 0.876 1.458 1.648 0.772 1.005 1.258 1.293 0.752 1.298 0.877 0.728
 1.219]
Ep 500 | Loss 3.41e+04 | Tau: [0.283 1.214 0.702 1.867 0.926 0.79  2.417 0.956 1.331 0.875 1.066 2.022
 0.317 0.946 1.606 1.946 0.789 1.095 1.475 1.451 0.78  1.349 0.948 0.738
 1.36 ]
Ep 600 | Loss 3.40e+04 | Tau: [0.246 1.663 0.944 2.003 0.976 0.827 2.578 1.084 1